# Activity: Maximizing Profit with Multiplicative Weights
In this graded activity, we use the multiplicative weights update algorithm to solve a linear _optimization_ problem: choosing a production mix that maximizes profit subject to resource limits. We build the optimizer on top of the feasibility solver and check it against an exact linear programming solver.

> __Learning Objectives.__
>
> By the end of this activity, you will be able to:
>
> * __Turn feasibility into optimization:__ Add an objective cut to the constraints and binary search over its value to maximize profit with the feasibility solver.
> * __Solve a production planning problem:__ Allocate a fixed production budget across products to maximize profit while respecting resource capacities.
> * __Validate against an exact solver:__ Compare the multiplicative weights optimum to the GLPK solution and confirm the objective values agree.

Let's get started!
___

## Background: optimization by binary search on feasibility
The feasibility solver answers whether a point exists on the simplex satisfying $\mathbf{A}\mathbf{x}\le\mathbf{b}$. To _maximize_ a linear objective $\mathbf{c}^{\top}\mathbf{x}$, we add the objective as a constraint and search for the best value it can take.

> For a trial value $v$, we ask whether a feasible point with $\mathbf{c}^{\top}\mathbf{x}\ge v$ exists. We write this as the packing row $-\mathbf{c}^{\top}\mathbf{x}\le -v$ and append it to $\mathbf{A}\mathbf{x}\le\mathbf{b}$. If the augmented problem is feasible, then $v$ is achievable and we raise the target; otherwise we lower it. Binary search on $v$ converges to the optimal objective value.

This logic is implemented in [the `solve(...)` method](src/Solve.jl) for [the `MyLinearProgramOptimizationProblem` type](src/Types.jl), which calls the feasibility solver repeatedly. Let's set up the production problem.
___

<div>
    <center>
        <img src="figs/production-tikz/production.svg" width="780"/>
    </center>
</div>

## The production mix problem
A plant makes four products from three shared resources, for example reactor time, feedstock, and separation capacity. We allocate a fixed total production of $\tau = 8$ batches across the four products. Producing one batch of product $i$ consumes $a_{ki}$ units of resource $k$, and each resource $k$ has a capacity $b_{k}$. Each batch of product $i$ earns a profit $c_{i}$.

> __Goal.__ Choose the production amounts $\mathbf{x}\in\Delta_{m} = \{\mathbf{x}\ge 0 : \sum_{i} x_{i} = \tau\}$ that maximize total profit $\mathbf{c}^{\top}\mathbf{x}$ while keeping every resource within capacity, $\mathbf{A}\mathbf{x}\le\mathbf{b}$.

The resource usage is $\mathbf{A}\in\mathbb{R}^{3\times 4}$, the capacities are $\mathbf{b} = (16, 20, 14)$, and the per-batch profits are $\mathbf{c} = (6, 5, 4, 7)$. Let's load the environment.
___

## Setup, Data, and Prerequisites
We include the `Include.jl` file to load the required packages and the module's local `src/` codes.

In [1]:
include("Include.jl"); # load my codes, packages, etc

## Task 1: Build the optimization problem
Construct [a `MyLinearProgramOptimizationProblem` instance](src/Types.jl) holding the resource matrix $\mathbf{A}$, the capacity vector $\mathbf{b}$, the profit vector $\mathbf{c}$, and the production budget $\tau$. Use [the `build(...)` method](src/Factory.jl) and store it in `production_problem`.

In [2]:
production_problem = let

    # resource usage A[k,i]: units of resource k used per batch of product i -
    A = [
        2.0  1.0  1.0  2.0 ;  # resource 1 (e.g., reactor time)
        1.0  2.0  3.0  1.0 ;  # resource 2 (e.g., feedstock)
        1.0  1.0  1.0  3.0 ;  # resource 3 (e.g., separation)
    ];
    b = [16.0, 20.0, 14.0]; # resource capacities
    c = [6.0, 5.0, 4.0, 7.0]; # per-batch profit
    τ = 8.0;                # total production budget: sum(x) = τ

    problem = build(MyLinearProgramOptimizationProblem, (
        A = A, b = b, c = c, τ = τ, ϵ = 0.004));
    problem; # return the problem
end;

Let's confirm the problem was built with the expected dimensions and data.

In [3]:
let
    @assert size(production_problem.A) == (3, 4)
    @assert length(production_problem.c) == 4
    @assert production_problem.τ == 8.0
    println("Task 1 checks passed.");
end

Task 1 checks passed.


## Task 2: Solve with multiplicative weights
Pass the `production_problem` to [the `solve(...)` method](src/Solve.jl). It binary searches the profit objective, calling the feasibility solver at each step, and returns the best production plan it finds. Store the result dictionary in `mwa_result`.

In [4]:
mwa_result = solve(production_problem; tol = 1e-3);

The returned plan should use the full production budget and respect every resource capacity. Let's verify.

In [5]:
let
    x = mwa_result["x"];
    A = production_problem.A;
    b = production_problem.b;
    τ = production_problem.τ;

    println("production plan x = ", round.(x, digits = 4));
    println("total production  = ", round(sum(x), digits = 4), "   (budget τ = ", τ, ")");
    println("MWA profit        = ", round(mwa_result["objective_value"], digits = 4));

    @assert isapprox(sum(x), τ; atol = 1e-6) # uses the full budget
    @assert all(A*x .<= b .+ 0.05)            # within every resource capacity
    println("Task 2 checks passed.");
end

production plan x = [4.9961, 0.0, 0.0, 3.0039]
total production  = 8.0   (budget τ = 8.0)
MWA profit        = 51.0039
Task 2 checks passed.


## Task 3: Compute the exact optimum with GLPK
To grade our approximate optimizer, we compute the exact optimum with the GLPK solver. The simplex constraint $\sum_{i} x_{i} = \tau$ is encoded as two inequalities, $\sum_{i} x_{i}\le\tau$ and $-\sum_{i} x_{i}\le -\tau$, appended to $\mathbf{A}\mathbf{x}\le\mathbf{b}$. We build [a `MyLinearProgrammingProblemModel`](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) and solve it with `constraints = :leq`. Store the result in `glpk_result`.

In [6]:
glpk_result = let
    A = production_problem.A;
    b = production_problem.b;
    c = production_problem.c;
    τ = production_problem.τ;
    m = size(A, 2);

    # encode sum(x) = τ as two ≤ rows: sum(x) ≤ τ and -sum(x) ≤ -τ -
    A_aug = vcat(A, ones(1, m), -ones(1, m));
    b_aug = vcat(b, τ, -τ);

    lp = build(MyLinearProgrammingProblemModel, (
        A = A_aug, b = b_aug, c = c, lb = zeros(m), ub = fill(τ, m)));
    result = VLDataScienceMachineLearningPackage.solve(lp; constraints = :leq);
    result; # return the GLPK result
end;

In [7]:
let
    println("GLPK plan   = ", round.(glpk_result["argmax"], digits = 4));
    println("GLPK profit = ", round(glpk_result["objective_value"], digits = 4));
end

GLPK plan   = [5.0, 0.0, 0.0, 3.0]
GLPK profit = 51.0


## Task 4: Compare the two solutions
A linear program can have more than one optimal corner, so the multiplicative weights plan and the GLPK plan need not be identical. The _optimal objective value_, however, is unique. We grade the approximate optimizer by comparing profit values: they should agree to within a small tolerance.

In [8]:
let
    mwa_profit = mwa_result["objective_value"];
    glpk_profit = glpk_result["objective_value"];
    gap = abs(mwa_profit - glpk_profit) / abs(glpk_profit);

    println("MWA profit   = ", round(mwa_profit, digits = 4));
    println("GLPK profit  = ", round(glpk_profit, digits = 4));
    println("relative gap = ", round(100*gap, digits = 3), " %");

    @assert gap < 0.02 # MWA optimum matches the exact optimum within 2%
    println("Task 4 checks passed — the multiplicative weights optimum matches GLPK.");
end

MWA profit   = 51.0039
GLPK profit  = 51.0
relative gap = 0.008 %
Task 4 checks passed — the multiplicative weights optimum matches GLPK.


### The production plan
Finally, let's display the multiplicative weights production plan: the number of batches of each product, the profit it contributes, and the cumulative profit. `Unhide` the code block below to see how we build the table.

In [9]:
let
    x = mwa_result["x"];
    c = production_problem.c;

    df = DataFrame();
    cumulative = 0.0;
    for i ∈ eachindex(x)
        cumulative += c[i]*x[i];
        push!(df, (product = "Product-$(i)", batches = x[i], profit = c[i]*x[i], cumulative = cumulative));
    end
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

 ----------- ------------- ------------- ------------
    product       batches        profit   cumulative 
     String       Float64       Float64      Float64 
 ----------- ------------- ------------- ------------
  Product-1       4.99608       29.9765      29.9765
  Product-2   6.61356e-14   3.30678e-13      29.9765
  Product-3   1.55494e-28   6.21974e-28      29.9765
  Product-4       3.00392       21.0275      51.0039
 ----------- ------------- ------------- ------------


___

## Summary
In this activity, we built a profit-maximizing optimizer from the feasibility solver and used it to choose a production mix, validating the result against an exact linear programming solver.

> __Key Takeaways:__
>
> * __Optimization from feasibility:__ Adding the objective as a constraint and binary searching its value turns the multiplicative weights feasibility solver into a linear program optimizer.
> * __Production planning:__ Allocating a fixed budget across products to maximize profit under resource limits is a linear program the method solves directly.
> * __Matching the exact optimum:__ The approximate optimal profit agrees with the GLPK solution, even when the chosen production plans differ, because the optimal objective value is unique.

The multiplicative weights feasibility solver, wrapped in a binary search on the objective, recovers the exact optimal profit for the production problem. This completes the progression from feasibility, to maximum throughput, to constrained profit maximization, all built on a single weight-update engine.
___